# QLoRA fine-tune: Qwen2.5-1.5B-Instruct for customer support

Run top to bottom on Colab Pro (A100 or L4) or on a free T4.
Nothing is mounted; every artifact is written under `train/runs/<run>/` and zipped at the end.

Order: GPU check, clone, install pinned versions, token, data, smoke run with a resume proof,
full run, per-checkpoint dev answers, curves, download.

Configs: `configs/train.yaml` (full grouped train split) and `configs/train-t4.yaml` (8,000 rows,
the free-T4 budget). Hyperparameters are fixed by `docs/dag/CONTRACTS.md` section 5.


## 1. GPU


In [ ]:
!nvidia-smi


## 2. Clone the repository

Set `BRANCH` to the tag or branch you want to reproduce. `REPO_URL` needs a token for a private repo.


In [ ]:
import os

REPO_URL = 'https://github.com/sharadja/ghl-support-slm.git'
BRANCH = 'main'
WORKDIR = '/content/ghl-support-slm'

if not os.path.isdir(WORKDIR):
    !git clone --branch $BRANCH --depth 1 $REPO_URL $WORKDIR
os.chdir(WORKDIR)
!git rev-parse HEAD


## 3. Install the pinned versions

Colab ships different builds; these are the versions in `configs/versions.json` and `uv.lock`.
Restarting the runtime after this cell is normal if Colab asks.


In [ ]:
!pip install -q \
  'transformers==5.16.1' \
  'peft==0.20.0' \
  'trl==1.12.0' \
  'bitsandbytes==0.50.2' \
  'accelerate>=0.34' \
  'datasets==5.0.1' \
  'sentence-transformers>=3.0' \
  'scikit-learn>=1.5' \
  'pyyaml>=6.0' \
  'matplotlib>=3.9'

import torch, transformers, peft, bitsandbytes
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__, 'peft', peft.__version__, 'bnb', bitsandbytes.__version__)


## 4. Hugging Face token

Add `HF_TOKEN` to Colab secrets (key icon in the left sidebar) with write access, then run this cell.
Without it the run still trains; it only skips the Hub push, with a warning.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab secrets')
except Exception as exc:
    print('no HF_TOKEN:', exc)
    print('training will run and checkpoints stay local')


## 5. Data

`data/raw/` and `data/processed/` are gitignored, so the split is regenerated here from the pinned
dataset revision. `data/prepare.py --audit-only --strict` then proves the hashes match `data/splits.json`.


In [ ]:
!python data/fetch.py
!python data/prepare.py
!python data/prepare.py --audit-only --strict


## 6. Smoke run: 20 steps, then a resume from step 10

This trains 20 steps on 64 rows, drops the later checkpoint, resumes from step 10 and checks that
steps 11 to 20 reproduce the original losses. `smoke.json` records the comparison; `check_run.py`
fails if it did not match. Do not start the full run until this passes.


In [ ]:
!python train/train.py --config configs/train.yaml --smoke --data-dir data/processed
!python tools/check_run.py train/runs/smoke-ghl-support-qlora --max-memory-gb 12


## 7. Full run

One epoch over the grouped train split, checkpoints at 50 percent and 100 percent plus every 100 steps.
Swap in `configs/train-t4.yaml` on a free T4. If the session drops, re-run this cell with `--resume`;
it continues from the newest checkpoint and replays the same data order.

Planning estimate, not a measurement: 30 to 60 minutes on an A100 or L4 for the full split,
and roughly 1 to 1.5 hours for 8,000 rows on a T4. Record the real number from `config.json`.


In [ ]:
CONFIG = 'configs/train.yaml'   # use configs/train-t4.yaml on a free T4
!python train/train.py --config $CONFIG
# If the session dropped mid-run, use this instead:
# !python train/train.py --config $CONFIG --resume


## 8. Dev answers for every checkpoint

Each checkpoint is merged into `artifacts/merged`, answered against the 54 dev items through the
transformers backend, and written to `train/runs/<run>/dev-<ckpt>-raw.jsonl`. The merged directory is
removed between checkpoints because `tools/merge.py` refuses to overwrite.


In [ ]:
import shutil, subprocess
import yaml
from pathlib import Path

run_name = yaml.safe_load(Path(CONFIG).read_text())['run_name']
run_dir = Path('train/runs') / run_name
RUN_DIR = str(run_dir)
checkpoints = sorted(
    (int(p.name.removeprefix('checkpoint-')), p)
    for p in run_dir.glob('checkpoint-*') if (p / 'adapter_config.json').is_file()
)
print('checkpoints:', [step for step, _ in checkpoints])

for step, folder in checkpoints:
    shutil.rmtree('artifacts/merged', ignore_errors=True)
    subprocess.run(['python', 'tools/merge.py', '--adapter', str(folder),
                    '--output', 'artifacts/merged'], check=True)
    out = run_dir / f'dev-checkpoint-{step}-raw.jsonl'
    subprocess.run(['python', 'eval/run.py', '--model', 'tuned', '--backend', 'transformers',
                    '--split', 'dev', '--device', 'cuda', '--output', str(out),
                    '--check-complete'], check=True)
    print('wrote', out)
shutil.rmtree('artifacts/merged', ignore_errors=True)


## 9. Curves, gate, download


In [ ]:
!python train/plot_curves.py $RUN_DIR
!python tools/check_run.py $RUN_DIR --max-memory-gb 12
!cat $RUN_DIR/config.json | python -c "import json,sys; d=json.load(sys.stdin); print({k: d[k] for k in ('run_name','device','wall_s','peak_memory_gb','final_step','git_sha')})"


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(f'/content/{run_dir.name}', 'zip', run_dir)
print('zipped', archive)
files.download(archive)


## What to paste back into the thread

- `config.json`: `wall_s`, `peak_memory_gb`, `final_step`, `git_sha`, `versions`.
- `smoke.json`: `ok` and `max_abs_diff`.
- The last few lines of `loss.csv`.
- The checkpoint list and the `dev-checkpoint-*-raw.jsonl` row counts.
- The zip, or the Hub repo URL if the push worked.
